# 03.01 Why Graphics Needs Cross Product

在 Chapter 02 中，我们已经学习了 dot product。

dot product 的核心作用是衡量两个方向之间的 alignment：

$$
u \cdot v = ||u|| ||v|| \cos(\theta)
$$

如果 u 和 v 都是单位向量，那么 dot product 直接等于两个方向夹角的 cos 值。

因此 dot product 很适合回答这类问题：

- 两个方向是否同向？
- 两个方向是否垂直？
- 表面是否朝向光源？
- 目标是否在相机前方？
- 点在某个方向上的投影长度是多少？

但是 dot product 不能回答另一个图形学中极其重要的问题：

```text
两个方向共同张成的平面，应该朝哪一侧？

## 03.01.01 Dot Product Is Not Enough

假设我们有两个三维向量 u 和 v。

dot product 给出的是一个 scalar：

$$
u \cdot v
$$

这个 scalar 告诉我们 u 和 v 的夹角关系。

但是它不会告诉我们：

- u 和 v 所在平面的法线方向；
- 三角形的正面朝向哪边；
- 顶点顺序是 clockwise 还是 counter-clockwise；
- 相机坐标系中的 right / up / forward 如何互相构造；
- 一个旋转方向是顺时针还是逆时针。

换句话说：

```text
dot product answers: how aligned are two directions?
cross product answers: what direction is perpendicular to both directions?

## 03.01.02 The Core Graphics Problem: Getting a Surface Normal

在图形学中，一个三角形通常由三个点表示：

$$
p_0,\ p_1,\ p_2
$$

为了计算这个三角形的表面法线，我们先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

这两条边都在三角形所在的平面上。

现在问题变成：

```text
如何找到一个同时垂直于 e1 和 e2 的方向？

## 03.01.03 A Concrete Numerical Example

考虑一个位于 xy 平面上的三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

构造两条边：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

根据右手坐标系约定：

$$
e_1 \times e_2=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

所以这个三角形的 normal 指向正 z 方向。

如果交换顺序：

$$
e_2 \times e_1=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

方向会完全反过来。

这说明 cross product 不满足交换律：

$$
u \times v=-(v \times u)
$$

## 03.01.04 Why Direction Matters in Graphics

cross product 的方向不是一个纯数学细节，而是直接影响图形学结果。

如果三角形 normal 方向错了，会影响：

1. lighting  
   表面可能被错误地认为背对光源。

2. back-face culling  
   本来应该显示的三角形可能被剔除。

3. shading  
   法线方向错误会导致明暗反转。

4. collision detection  
   接触面方向错误会导致响应方向错误。

5. camera basis  
   right、up、forward 构造顺序错误会导致相机坐标系翻转。

因此 cross product 的方向约定必须和坐标系约定一起理解。

## 03.01.05 Cross Product as Orientation Information

dot product 的结果是 scalar。

cross product 的结果是 vector。

这两者的几何意义不同：

```text
dot product:
two vectors → scalar
measures alignment

cross product:
two 3D vectors → vector
produces a perpendicular direction

## 03.01.06 Graphics Applications Preview

后续章节中，cross product 会反复出现。

第一类应用是 surface normal。

给定三角形三个点：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

$$
n=\operatorname{normalize}(e_1 \times e_2)
$$

第二类应用是 camera basis。

如果我们知道 camera forward 和 world up，可以构造 camera right：

$$
r=\operatorname{normalize}(f \times up)
$$

然后再构造真正的 camera up：

$$
u=\operatorname{normalize}(r \times f)
$$

第三类应用是 back-face culling。

如果 triangle normal 和 view direction 的关系表明三角形背对相机，那么可以不绘制它。

第四类应用是面积。

cross product 的长度和两个向量张成的平行四边形面积有关：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

三角形面积则是它的一半：

$$
A=\frac{1}{2}||e_1 \times e_2||
$$

## 03.01.07 NumPy Example

下面先用 NumPy 验证最基本的 cross product 方向。

我们使用：

```text
x × y = z
y × x = -z

In [ ]:
import numpy as np

x = np.array([1.0, 0.0, 0.0])
y = np.array([0.0, 1.0, 0.0])
z = np.array([0.0, 0.0, 1.0])

cross_xy = np.cross(x, y)
cross_yx = np.cross(y, x)

print("x cross y:", cross_xy)
print("y cross x:", cross_yx)
print("Expected z:", z)
print("Expected -z:", -z)

x cross y: [0. 0. 1.]
y cross x: [ 0.  0. -1.]
Expected z: [0. 0. 1.]
Expected -z: [-0. -0. -1.]


## 03.01.08 NumPy Example: Triangle Normal

现在考虑三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

先构造边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

再计算：

$$
n=e_1 \times e_2
$$

In [2]:
def normalize(v):
    """归一化向量；zero vector 不能归一化。"""
    v = np.asarray(v, dtype=float)
    length = np.linalg.norm(v)

    if np.isclose(length, 0.0):
        raise ValueError("Cannot normalize a zero vector.")

    return v / length


p0 = np.array([0.0, 0.0, 0.0])
p1 = np.array([1.0, 0.0, 0.0])
p2 = np.array([0.0, 1.0, 0.0])

e1 = p1 - p0
e2 = p2 - p0

n = np.cross(e1, e2)
n_hat = normalize(n)

print("e1:", e1)
print("e2:", e2)
print("normal:", n)
print("unit normal:", n_hat)

e1: [1. 0. 0.]
e2: [0. 1. 0.]
normal: [0. 0. 1.]
unit normal: [0. 0. 1.]


## 03.01.09 What Happens If We Reverse Vertex Order?

如果交换 p1 和 p2，相当于改变三角形的 winding order。

原来是：

$$
n=(p_1-p_0) \times (p_2-p_0)
$$

交换后是：

$$
n'=(p_2-p_0) \times (p_1-p_0)
$$

因为 cross product 反交换：

$$
n'=-n
$$

所以 normal direction 会翻转。

这就是 winding order 会影响 triangle normal 的原因。

## 03.01.11 Exercises

### Exercise 1

给定：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请判断：

$$
u \times v
$$

的方向。

再判断：

$$
v \times u
$$

的方向。

---

### Exercise 2

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

并判断：

$$
e_1 \times e_2
$$

的方向。

---

### Exercise 3

如果一个三角形的顶点顺序从：

$$
p0 → p1 → p2
$$

改成：
$$
p0 → p2 → p1
$$
请说明 normal direction 会发生什么变化。

# 03.02 Cross Product Definition

上一节我们已经知道，cross product 用来从两个 3D 向量构造一个新的方向。

这个新方向有两个关键性质：

1. 它垂直于第一个输入向量；
2. 它垂直于第二个输入向量。

也就是说，如果输入是 u 和 v，那么 cross product 输出的是：

$$
u \times v
$$

这个结果不是 scalar，而是一个 vector。

这和 dot product 完全不同：

```text
dot product:
u · v → scalar

cross product:
u × v → vector

## 03.02.01 Cross Product Only Makes Sense Here as a 3D Operation

在本课程的图形学语境中，我们主要讨论 3D cross product。

给定两个三维向量：

$$
u=
\begin{bmatrix}
u_x \\
u_y \\
u_z
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
v_x \\
v_y \\
v_z
\end{bmatrix}
$$

它们的 cross product 仍然是一个三维向量：

$$
u \times v=
\begin{bmatrix}
? \\
? \\
?
\end{bmatrix}
$$

这个输出向量的方向由右手定则决定。

它的长度和 u、v 张成的面积有关。

这一节先重点掌握代数公式。

## 03.02.02 Algebraic Definition

给定：

$$
u=
\begin{bmatrix}
u_x \\
u_y \\
u_z
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
v_x \\
v_y \\
v_z
\end{bmatrix}
$$

cross product 定义为：

$$
u \times v=
\begin{bmatrix}
u_y v_z-u_z v_y \\
u_z v_x-u_x v_z \\
u_x v_y-u_y v_x
\end{bmatrix}=
\begin{bmatrix}
D_{yz} \\
D_{zx} \\
D_{xy}
\end{bmatrix}
$$

也就是说：

$$
(u \times v)_x=u_y v_z-u_z v_y
$$

$$
(u \times v)_y=u_z v_x-u_x v_z
$$

$$
(u \times v)_z=u_x v_y-u_y v_x
$$

注意第二个分量很容易写错。

它是：

$$
u_z v_x-u_x v_z
$$

不是：

$$
u_x v_z-u_z v_x
$$

## 03.02.03 Concrete Example: x Cross y

令：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

根据公式：

$$
x \times y=
\begin{bmatrix}
0 \cdot 0-0 \cdot 1 \\
0 \cdot 0-1 \cdot 0 \\
1 \cdot 1-0 \cdot 0
\end{bmatrix}
$$

所以：

$$
x \times y=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

这正好是 z 方向。

因此在右手坐标系中：

$$
x \times y=z
$$

## 03.02.04 Reversing the Order

现在交换顺序：

$$
y \times x
$$

其中：

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

根据公式：

$$
y \times x=
\begin{bmatrix}
1 \cdot 0-0 \cdot 0 \\
0 \cdot 1-0 \cdot 0 \\
0 \cdot 0-1 \cdot 1
\end{bmatrix}
$$

所以：

$$
y \times x=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

也就是说：

$$
y \times x=-z
$$

因此：

$$
u \times v=-(v \times u)
$$

cross product 不满足交换律。

## 03.02.05 Cross Product Is Orthogonal to Both Inputs

cross product 的核心性质是：

$$
u \times v \perp u
$$

同时：

$$
u \times v \perp v
$$

换成 dot product 表示就是：

$$
(u \times v) \cdot u=0
$$

$$
(u \times v) \cdot v=0
$$

这就是为什么 cross product 可以用来计算 surface normal。

如果三角形的两条边是：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

那么：

$$
n=e_1 \times e_2
$$

会同时垂直于 e1 和 e2。

因此 n 垂直于三角形所在平面。

## 03.02.06 Numerical Example with Non-Axis Vectors

令：

$$
u=
\begin{bmatrix}
1 \\
2 \\
3
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
4 \\
5 \\
6
\end{bmatrix}
$$

根据公式：

$$
u \times v=
\begin{bmatrix}
2 \cdot 6-3 \cdot 5 \\
3 \cdot 4-1 \cdot 6 \\
1 \cdot 5-2 \cdot 4
\end{bmatrix}
$$

所以：

$$
u \times v=
\begin{bmatrix}
-3 \\
6 \\
-3
\end{bmatrix}
$$

验证它和 u 垂直：

$$
(u \times v) \cdot u=(-3)\cdot 1+6\cdot 2+(-3)\cdot 3
$$

$$
(u \times v) \cdot u=0
$$

验证它和 v 垂直：

$$
(u \times v) \cdot v=(-3)\cdot 4+6\cdot 5+(-3)\cdot 6
$$

$$
(u \times v) \cdot v=0
$$

## 03.02.07 NumPy Implementation: Using np.cross

NumPy 已经提供了 cross product 函数：

```text
np.cross(u, v)

In [3]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])

cross_uv = np.cross(u, v)
cross_vu = np.cross(v, u)

print("u cross v:", cross_uv)
print("v cross u:", cross_vu)
print("-(v cross u):", -cross_vu)

print("Are they opposite?", np.allclose(cross_uv, -cross_vu))

u cross v: [-3.  6. -3.]
v cross u: [ 3. -6.  3.]
-(v cross u): [-3.  6. -3.]
Are they opposite? True


## 03.02.10 Graphics Connection: Triangle Normal

cross product 最直接的图形学应用是计算三角形法线。

给定三角形三个点：

$$
p_0,\ p_1,\ p_2
$$

先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

然后：

$$
n=e_1 \times e_2
$$

最后归一化：

$$
\hat{n}=\frac{n}{||n||}
$$

这里必须注意：

```text
e1 × e2 和 e2 × e1 的方向相反。

## 03.02.11 Degenerate Triangle Case

如果三角形的三个点共线，那么两条边不能张成一个真正的平面。

例如：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

此时：

$$
e_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

e1 和 e2 共线。

所以：

$$
e_1 \times e_2=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

zero vector 不能 normalized。这类三角形称为 degenerate triangle，在图形学中，它没有有效的 surface normal。

## 03.02.14 Exercises

### Exercise 1

给定：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

请计算：

$$
u \times v
$$

并判断结果指向哪个坐标轴方向。

---

### Exercise 2

给定：

$$
u=
\begin{bmatrix}
1 \\
2 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
3 \\
1 \\
0
\end{bmatrix}
$$

请手动计算：

$$
u \times v
$$

并说明为什么结果只在 z 方向上有分量。

---

### Exercise 3

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

以及：

$$
n=e_1 \times e_2
$$

判断这个 normal 指向正 z 方向还是负 z 方向。

---

### Exercise 4

为什么下面三个点不能定义出一个有效的三角形 normal？

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
1 \\
1
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
2 \\
2
\end{bmatrix}
$$

请从 cross product 的角度解释。

---

### Exercise 5

用 NumPy 实现一个函数：

```text
triangle_normal(p0, p1, p2)

# 03.03 Right-Hand Rule and Direction

cross product 最容易出错的地方不是数值计算，而是方向。

对于 dot product：

$$
u \cdot v=v \cdot u
$$

交换顺序不会改变结果。

但对于 cross product：

$$
u \times v=-(v \times u)
$$

交换顺序会让方向完全反过来。

因此，在图形学中使用 cross product 时，必须同时关注两件事：

1. 输入向量的顺序；
2. 当前使用的坐标系约定。

本项目采用 right-handed coordinate system。

在这个约定下：

$$
x \times y=z
$$

## 03.03.01 The Right-Hand Rule

右手定则用于判断 cross product 的方向。

对于：

$$
u \times v
$$

可以这样理解：

```text
右手四指从 u 的方向弯向 v 的方向，
大拇指指向的方向就是 u × v 的方向。

## 03.03.02 Standard Axes in a Right-Handed Coordinate System

在 right-handed coordinate system 中，标准基向量满足：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
z=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

并且：

$$
x \times y=z
$$

$$
y \times z=x
$$

$$
z \times x=y
$$

这三个关系可以看作右手坐标系中的正向循环。

反过来：

$$
y \times x=-z
$$

$$
z \times y=-x
$$

$$
x \times z=-y
$$

所以 cross product 的顺序不能随便交换。

## 03.03.03 Concrete Example: x Cross z

令：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
z=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

根据 right-handed coordinate system 的循环关系：

$$
z \times x=y
$$

因此反过来：

$$
x \times z=-y
$$

所以：

$$
x \times z=
\begin{bmatrix}
0 \\
-1 \\
0
\end{bmatrix}
$$

这个例子很重要，因为很多人会直觉上误以为 x × z 等于 y。

但在右手坐标系下，正确结果是：

$$
x \times z=-y
$$

## 03.03.04 Numerical Verification with the Formula

用公式验证：

$$
u \times v=
\begin{bmatrix}
u_y v_z-u_z v_y \\
u_z v_x-u_x v_z \\
u_x v_y-u_y v_x
\end{bmatrix}
$$

令：

$$
u=x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=z=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

则：

$$
x \times z=
\begin{bmatrix}
0 \cdot 1-0 \cdot 0 \\
0 \cdot 0-1 \cdot 1 \\
1 \cdot 0-0 \cdot 0
\end{bmatrix}
$$

所以：

$$
x \times z=
\begin{bmatrix}
0 \\
-1 \\
0
\end{bmatrix}
$$

这和右手定则一致。

## 03.03.05 Orientation of a Triangle

cross product 的方向会决定 triangle normal 的方向。

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

构造两条边：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

因此：

$$
n=e_1 \times e_2=x \times y=z
$$

所以 normal 指向正 z 方向。

如果交换 p1 和 p2：

$$
e_1'=p_2-p_0
$$

$$
e_2'=p_1-p_0
$$

则：

$$
n'=e_1' \times e_2'=y \times x=-z
$$

normal 会翻转到负 z 方向。

## 03.03.06 Winding Order and Normal Direction

三角形顶点顺序通常称为 winding order。

常见的 winding order 包括：

```text
counter-clockwise winding
clockwise winding

## 03.03.09 Graphics Application: Camera Basis Direction

cross product 也常用于构造 camera basis。

假设我们有：

- camera forward direction；
- approximate world up direction。

在右手坐标系中，如果 camera forward 指向屏幕里面，也就是类似 OpenGL 中的负 z 方向：

$$
f=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

world up 为：

$$
up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

那么可以用：

$$
r=\operatorname{normalize}(f \times up)
$$

得到 right direction。

计算结果是：

$$
r=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

然后再用：

$$
u=\operatorname{normalize}(r \times f)
$$

得到真正的 camera up direction。

注意：camera basis 的 cross product 顺序和你采用的 forward 方向约定有关。后续讲 View Matrix 时会专门处理这个问题。

## 03.03.11 Why This Matters for Lighting

在 Lambert lighting 中，我们使用：

$$
I=\max(0,\ n \cdot l)
$$

其中 n 是 surface normal，l 是从 surface point 指向 light source 的单位方向。

如果 triangle normal 因为顶点顺序错误而反向，那么原本应该被照亮的面可能变暗。

例如：

$$
n=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

光照方向为：

$$
l=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

那么：

$$
n \cdot l=1
$$

表面完全朝向光源。

如果 normal 翻转：

$$
n'=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

则：

$$
n' \cdot l=-1
$$

Lambert intensity 会变成：

$$
I=\max(0,\ -1)=0
$$

这说明 normal direction 错误会直接造成 lighting 错误。

## 03.03.13 Exercises

### Exercise 1

在 right-handed coordinate system 中，判断下面结果：

$$
x \times y
$$

$$
y \times z
$$

$$
z \times x
$$

分别指向哪个坐标轴方向。

---

### Exercise 2

在 right-handed coordinate system 中，判断下面结果：

$$
y \times x
$$

$$
z \times y
$$

$$
x \times z
$$

分别指向哪个坐标轴方向。

---

### Exercise 3

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

以及：

$$
n=e_1 \times e_2
$$

判断 normal 指向正 y 方向还是负 y 方向。

---

### Exercise 4

如果一个三角形的 normal 指向错误，可能会对 lighting 和 back-face culling 造成什么影响？

请分别用一句话说明。

---

### Exercise 5

假设：

$$
f=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

$$
up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请计算：

$$
f \times up
$$

并判断它是否指向正 x 方向。

---

### Exercise 6

用 NumPy 验证：

$$
u \times v=-(v \times u)
$$

任选两个非零且不共线的 3D 向量即可。

# 03.04 Cross Product Magnitude and Area

上一节我们已经学习：

$$
u \times v
$$

的方向由 right-hand rule 决定。

但 cross product 不只包含方向信息，也包含大小信息。

它的长度满足：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

其中 theta 是 u 和 v 之间的夹角。

这个公式说明：

```text
cross product 的长度 = 两个向量张成的平行四边形面积

## 03.04.01 From Direction to Magnitude

cross product 的结果是一个 vector。

这个 vector 有两个方面：

1. direction  
   由 right-hand rule 决定。

2. magnitude  
   由两个输入向量张成的面积决定。

也就是说：

$$
u \times v
$$

不仅告诉我们“垂直方向朝哪边”，还告诉我们“这两个向量张开的面积有多大”。

如果 u 和 v 几乎同向，那么它们张开的面积很小。

如果 u 和 v 垂直，那么它们张开的面积最大。

如果 u 和 v 完全共线，那么它们张不开一个平面，面积为 0。

## 03.04.02 Cross Product Magnitude Formula

cross product 的长度公式是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

这里：

- ||u|| 是 u 的长度；
- ||v|| 是 v 的长度；
- theta 是 u 和 v 的夹角；
- sin(theta) 控制两个向量张开的程度。

这个公式和 dot product 的角度公式很像。

dot product 是：

$$
u \cdot v=||u|| ||v|| \cos(\theta)
$$

cross product magnitude 是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

所以可以这样理解：

```text
dot product uses cos(theta): measures alignment
cross product magnitude uses sin(theta): measures area / perpendicular spread

## 03.04.02 Cross Product Magnitude Formula

cross product 的长度公式是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

这里：

- ||u|| 是 u 的长度；
- ||v|| 是 v 的长度；
- theta 是 u 和 v 的夹角；
- sin(theta) 控制两个向量张开的程度。

这个公式和 dot product 的角度公式很像。

dot product 是：

$$
u \cdot v=||u|| ||v|| \cos(\theta)
$$

cross product magnitude 是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

所以可以这样理解：

```text
dot product uses cos(theta): measures alignment
cross product magnitude uses sin(theta): measures area / perpendicular spread

## 03.04.04 Triangle Area

三角形面积是对应平行四边形面积的一半。

如果三角形由三个点 p0, p1, p2 定义，先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

这两条边张成一个平行四边形。

平行四边形面积是：

$$
||e_1 \times e_2||
$$

因此三角形面积是：

$$
A=\frac{1}{2}||e_1 \times e_2||
$$

这就是 cross product 在 mesh processing、collision detection、geometry analysis 中经常出现的原因。

## 03.04.05 Concrete Example: Perpendicular Vectors

令：

$$
u=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

这两个向量互相垂直，所以：

$$
\theta=90^\circ
$$

因此：

$$
\sin(\theta)=1
$$

cross product 为：

$$
u \times v=
\begin{bmatrix}
0 \\
0 \\
6
\end{bmatrix}
$$

所以：

$$
||u \times v||=6
$$

从面积角度看，u 和 v 张成的平行四边形是一个 2 × 3 的矩形。

因此面积也是：

$$
2 \times 3=6
$$

如果把它看作三角形的两条边，那么三角形面积是：

$$
A=\frac{1}{2}\times 6=3
$$

## 03.04.06 Concrete Example: Non-Perpendicular Vectors

令：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
1 \\
1 \\
0
\end{bmatrix}
$$

这两个向量的夹角是 45°。

先计算 cross product：

$$
u \times v=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

所以：

$$
||u \times v||=1
$$

再用长度公式验证。

u 的长度是：

$$
||u||=1
$$

v 的长度是：

$$
||v||=\sqrt{2}
$$

夹角为：

$$
\theta=45^\circ
$$

所以：

$$
||u|| ||v|| \sin(\theta)=1 \cdot \sqrt{2} \cdot \frac{\sqrt{2}}{2}
$$

得到：

$$
||u|| ||v|| \sin(\theta)=1
$$

这和 cross product 的长度一致。

## 03.04.08 Proof Sketch

cross product 的长度公式可以通过下面恒等式理解：

$$
||u \times v||^2=||u||^2||v||^2-(u \cdot v)^2
$$

又因为：

$$
u \cdot v=||u|| ||v|| \cos(\theta)
$$

所以：

$$
(u \cdot v)^2=||u||^2||v||^2\cos^2(\theta)
$$

代入得到：

$$
||u \times v||^2=||u||^2||v||^2-||u||^2||v||^2\cos^2(\theta)
$$

整理得到：

$$
||u \times v||^2=||u||^2||v||^2(1-\cos^2(\theta))
$$

因为：

$$
1-\cos^2(\theta)=\sin^2(\theta)
$$

所以：

$$
||u \times v||^2=||u||^2||v||^2\sin^2(\theta)
$$

两边开平方，得到：

$$
||u \times v||=||u|| ||v|| |\sin(\theta)|
$$

当 theta 取 0 到 pi 之间的夹角时，sin(theta) 非负，因此可以写成：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

## 03.04.13 Graphics Application: Mesh Area and Degenerate Triangles

cross product magnitude 在图形学中有很多实际用途。

第一，计算 triangle area。

对于 mesh 中的每个三角形：

$$
A_i=\frac{1}{2}||e_{1,i} \times e_{2,i}||
$$

整个 mesh 的表面积可以近似为所有三角形面积之和：

$$
A_{\text{mesh}}=\sum_i A_i
$$

第二，检测 degenerate triangles。

如果：

$$
||e_1 \times e_2|| \approx 0
$$

说明三角形面积接近 0。

这通常意味着：

- 三个顶点共线；
- 两个顶点重合；
- 三角形极度扁平；
- normal 无法稳定计算；
- shading 或 collision 可能出现异常。

第三，normal 计算前的有效性检查。

在计算：

$$
\hat{n}=\frac{e_1 \times e_2}{||e_1 \times e_2||}
$$

之前，必须检查：

$$
||e_1 \times e_2|| \ne 0
$$

否则就是 normalize zero vector。

## 03.04.14 Graphics Application: Area-Weighted Normals

在真实 mesh 中，一个顶点通常被多个三角形共享。

为了计算 vertex normal，可以把相邻三角形的 face normals 加权平均。

一种常见方法是 area-weighted normal。

如果某个 face 的未归一化法线是：

$$
n_i=e_{1,i} \times e_{2,i}
$$

它的长度正比于三角形面积：

$$
||n_i||=2A_i
$$

因此，直接累加未归一化的 face normal：

$$
n_{\text{vertex}}=\sum_i n_i
$$

相当于让面积更大的三角形对 vertex normal 贡献更大。

最后再 normalize：

$$
\hat{n}_{\text{vertex}}=\operatorname{normalize}(n_{\text{vertex}})
$$

这个思想在 smooth shading 中非常重要。

现在不需要完整实现，后续讲 lighting and shading 时会正式展开。